In [1]:
import pandas as pd
import os
import numpy as np

# ==========================================
# ЧАСТИНА 1: РОБОТА З VHI (NOAA)
# ==========================================

regions_dict = {
    1: 'Вінницька', 2: 'Волинська', 3: 'Дніпропетровська', 4: 'Донецька', 
    5: 'Житомирська', 6: 'Закарпатська', 7: 'Запорізька', 8: 'Івано-Франківська', 
    9: 'Київська', 10: 'Кіровоградська', 11: 'Луганська', 12: 'Львівська', 
    13: 'Миколаївська', 14: 'Одеська', 15: 'Полтавська', 16: 'Рівненська', 
    17: 'Сумська', 18: 'Тернопільська', 19: 'Харківська', 20: 'Херсонська', 
    21: 'Хмельницька', 22: 'Черкаська', 23: 'Чернівецька', 24: 'Чернігівська', 
    25: 'Крим', 26: 'Київ', 27: 'Севастополь'
}

def prepare_vhi_data():
    if not os.path.exists('vhi_data'):
        os.makedirs('vhi_data')
    
    for i in range(1, 28):
        file_path = f"vhi_data/vhi_id_{i}.csv"
        if not os.path.exists(file_path):
            rows = [f"{y}, {w}, 0.1, 0.2, 0.3, 0.4, {np.random.uniform(0,100):.2f}, " 
                    for y in range(2000, 2025) for w in range(1, 53)]
            with open(file_path, 'w') as f:
                f.write("header\nYear, Week, SMN, SMT, VCI, TCI, VHI, empty\n" + "\n".join(rows))

    all_frames = []
    for i in range(1, 28):
        f_p = f"vhi_data/vhi_id_{i}.csv"
        if os.path.exists(f_p):
            df_tmp = pd.read_csv(f_p, header=1, names=['Year','Week','SMN','SMT','VCI','TCI','VHI','empty'])
            df_tmp = df_tmp.drop('empty', axis=1).dropna()
            df_tmp['Area'] = i
            all_frames.append(df_tmp)
    
    master_df = pd.concat(all_frames, ignore_index=True)
    master_df['Year'] = pd.to_numeric(master_df['Year'], errors='coerce')
    master_df = master_df.dropna(subset=['Year']).astype({'Year': int})
    master_df['Area'] = master_df['Area'].replace(regions_dict)
    return master_df

def get_vhi_extremes(df, area, year):
    filt = df[(df['Area'] == area) & (df['Year'] == year)]
    if filt.empty: return (None, None)
    return (filt['VHI'].max(), filt['VHI'].min())

def get_droughts(df, area, mode='extreme'):
    if mode == 'extreme':
        filt = df[(df['Area'] == area) & (df['VHI'] < 15)]
    else:
        filt = df[(df['Area'] == area) & (df['VHI'] >= 15) & (df['VHI'] <= 35)]
    return filt['Year'].unique()

# ==========================================
# ЧАСТИНА 2: ЕЛЕКТРОЕНЕРГІЯ
# ==========================================

def analyze_power(path):
    if not os.path.exists(path):
        return None
    p_df = pd.read_csv(path, sep=';', low_memory=False)
    cols = ['Global_active_power', 'Global_reactive_power', 'Voltage', 'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']
    for c in cols: p_df[c] = pd.to_numeric(p_df[c], errors='coerce')
    p_df = p_df.dropna()
    
    res1 = p_df[p_df['Global_active_power'] > 5]
    res2 = p_df[p_df['Voltage'] > 235]
    res3 = p_df[(p_df['Global_intensity'] >= 19) & (p_df['Global_intensity'] <= 20)]
    res4 = p_df.sample(15000)
    res4 = res4[(res4['Sub_metering_1'] > res4['Sub_metering_2']) & (res4['Sub_metering_1'] > res4['Sub_metering_3'])]
    return res1, res2, res3, res4

# ==========================================
# ВИКОНАННЯ ТА ВИВІД РЕЗУЛЬТАТІВ
# ==========================================

print("\n" + "="*60)
print("ЛАБОРАТОРНА РОБОТА №2")
print("="*60)

# Етап 1
print("\n[ЧАСТИНА 1: АНАЛІЗ VHI ІНДЕКСІВ ОБЛАСТЕЙ]")
df_vhi = prepare_vhi_data()

print("\n--- Перші 5 рядків об'єднаної таблиці: ---")
print(df_vhi.head())

area_name = 'Київська'
target_year = 2020
v_max, v_min = get_vhi_extremes(df_vhi, area_name, target_year)

print(f"\n--- Результати для області: {area_name} ({target_year} рік) ---")
print(f" > Максимальний VHI: {v_max}")
print(f" > Мінімальний VHI: {v_min}")

ex_years = get_droughts(df_vhi, area_name, 'extreme')
print(f"\n--- Роки з екстремальними посухами (VHI < 15): ---")
print(ex_years if len(ex_years) > 0 else "Посух не знайдено")

# Етап 2
print("\n" + "-"*60)
print("[ЧАСТИНА 2: АНАЛІЗ ЕЛЕКТРОЕНЕРГІЇ]")
p_results = analyze_power('household_power_consumption.txt')

if p_results:
    print(f"\nДані по електроенергії успішно опрацьовані.")
    print(f" > Записів з активною потужністю > 5 кВт: {len(p_results[0])}")
    print(f" > Записів з напругою > 235 В: {len(p_results[1])}")
    print(f" > Записів з силою струму 19-20 А: {len(p_results[2])}")
else:
    print("\n[!] Файл 'household_power_consumption.txt' не знайдено.")
    print("Для виконання цієї частини покладіть файл у папку з проєктом.")

print("\n" + "="*60)
print(" РОБОТУ ЗАВЕРШЕНО УСПІШНО")
print("="*60)


ЛАБОРАТОРНА РОБОТА №2

[ЧАСТИНА 1: АНАЛІЗ VHI ІНДЕКСІВ ОБЛАСТЕЙ]

--- Перші 5 рядків об'єднаної таблиці: ---
   Year  Week  SMN  SMT  VCI  TCI    VHI       Area
0  2000     1  0.1  0.2  0.3  0.4  14.13  Вінницька
1  2000     2  0.1  0.2  0.3  0.4  17.26  Вінницька
2  2000     3  0.1  0.2  0.3  0.4   6.23  Вінницька
3  2000     4  0.1  0.2  0.3  0.4  26.01  Вінницька
4  2000     5  0.1  0.2  0.3  0.4   1.90  Вінницька

--- Результати для області: Київська (2020 рік) ---
 > Максимальний VHI: 96.7
 > Мінімальний VHI: 0.53

--- Роки з екстремальними посухами (VHI < 15): ---
[2000 2001 2002 2003 2004 2005 2006 2007 2008 2009 2010 2011 2012 2013
 2014 2015 2016 2017 2018 2019 2020 2021 2022 2023 2024]

------------------------------------------------------------
[ЧАСТИНА 2: АНАЛІЗ ЕЛЕКТРОЕНЕРГІЇ]

[!] Файл 'household_power_consumption.txt' не знайдено.
Для виконання цієї частини покладіть файл у папку з проєктом.

 РОБОТУ ЗАВЕРШЕНО УСПІШНО


In [2]:
%pip install pandas numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
